In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pandas.api.types import CategoricalDtype

# Detect the repository root
current_dir = Path.cwd()

if (current_dir / "data").exists():
    repository_root = current_dir
elif (current_dir.parent / "data").exists():
    repository_root = current_dir.parent
else:
    raise FileNotFoundError(
        "The repository data directory could not be found. "
        "Run the notebook from the repository root or from the notebooks directory."
    )

# Dataset path
data_path = repository_root / "data" / "Dataset_Consumo_Label_v1.xlsx"

# Verify that the dataset exists
if not data_path.exists():
    raise FileNotFoundError(
        "Dataset not found. Place 'Dataset_Consumo_Label_v1.xlsx' "
        "inside the repository data directory."
    )

# Load the dataset
df = pd.read_excel(data_path)

# Verify the required column
required_column = "Consumo"

if required_column not in df.columns:
    raise KeyError(
        f"The dataset must contain a column named '{required_column}'."
    )

# Remove rows without consumption data
df = df.dropna(subset=[required_column]).copy()

# Consumption thresholds used to generate the labels
lower_threshold = 0.36
upper_threshold = 6.29
anomaly_threshold = 8.00

# Assign the daily-consumption class
def classify_consumption(consumption):
    if consumption < lower_threshold:
        return "Low Consumption"
    elif consumption <= upper_threshold:
        return "Normal Consumption"
    elif consumption <= anomaly_threshold:
        return "High Consumption"
    else:
        return "Anomalous Activity"


df["Final_Label"] = df["Consumo"].apply(classify_consumption)

# Define the logical order of the consumption classes
label_order = [
    "Low Consumption",
    "Normal Consumption",
    "High Consumption",
    "Anomalous Activity"
]

categorical_type = CategoricalDtype(
    categories=label_order,
    ordered=True
)

df["Final_Label"] = df["Final_Label"].astype(categorical_type)

# Create the boxplot
fig, ax = plt.subplots(figsize=(8, 6))

df.boxplot(
    column="Consumo",
    by="Final_Label",
    grid=True,
    showfliers=False,
    patch_artist=True,
    ax=ax
)

# Axis labels and formatting
ax.set_xlabel("Daily Consumption Class", fontsize=12)
ax.set_ylabel("Water Consumption (m³)", fontsize=12)

ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(
    ["Low / None", "Normal", "High", "Anomalous"],
    fontsize=11
)

ax.tick_params(axis="y", labelsize=11)
ax.grid(True)

# Remove the automatic pandas titles
ax.set_title("")
fig.suptitle("")

fig.tight_layout()
plt.show()